# Main for Data Analysis 

Filters analyses and plots track and tour data

This notebook is to be executed after data/data_handling/data-aquisition.ipynb and before main_energy_sim.ipynb

### Imports 

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import seaborn as sns

import data.data_handling.data_visualization as dv
import data.data_handling.data_processing as dp
from data.data_handling.data_processing import colors_fleets, alpha_major, alpha_minor
import data.data_handling.sequential_analysis as sq

## Preprocessing 

### Data filtering
Load and preprocess the raw trips data.

Preprocessing includes, filtering out tracks shorter than 1 km, with avg. speeds > max. speed and with negative durations. Additioally adds forwarder data to each trip.

In [ ]:
# Define LOCATIONS
LOCATIONS = dp.LOCATIONS

# Load data
df_trips_unfiltered, df_fleet= dp.load_data()
# Preprocess data
df_trips = dp.preprocess_trips_data(df_trips_unfiltered, df_fleet)

# Fixes tours that 'spill over' into the next tour and cause tracks to be assigned to the wrong track
# This is a problem in the raw data (df_trips_unfiltered)
df_trips = dp.fix_faulty_tour_assignment(df_trips, df_trips_unfiltered)

# Fixes assigns a locations that corresponds to a home base to the end of each tour, if it does not
# do so anyway. Usually because the last track of a tour has been filtered out in preprocess_trips_data()
# If new home bases are addded, update the homebase_map in fix_faulty_tour_endings()
df_trips = dp.fix_faulty_tour_endings(df_trips, df_trips_unfiltered)
#df_trips.to_csv('./input/stations/tracks_filtered.csv')

### Process truck occupations and resample dataframes

In [ ]:
df_stops = dp.process_stops_data(df_trips)

# Cumulative time spent driving or in each type of location for each truck
df_rt_joined_plot = dp.calculate_rest_times_and_driving(df_stops, df_trips)

# Resample occupation data
# shows time spent (in min) not occurencess
truck_day_occ = dp.resample_occupation_data(df_stops, df_trips)

### Create tour statistics 

Aggregate all tracks that belong to a single tour into a row 

Creates: input/stations/tours.csv

In [ ]:
df_tours = dp.aggregate_tours(df_trips, save=True)

# Calculate the percentage of tours with distance_km < 300, < 375, < 500, < 1000
distances = [300, 375, 500, 1000]
percentages = {dist: (df_tours[df_tours['distance_km'] < dist].shape[0] / df_tours.shape[0]) * 100 for dist in distances}

for dist, perc in percentages.items():
    print(f"Tours under {dist} km: {perc:.2f}%")

print('\n')  
# Calculate the percentage of tours with distance_km < 300, < 375, < 500, < 1000 per freight forwarder
ff_percentages = {}
for ff in df_tours['freight_forwarder'].unique():
    ff_data = df_tours[df_tours['freight_forwarder'] == ff]
    ff_percentages[ff] = {dist: (ff_data[ff_data['distance_km'] < dist].shape[0] / ff_data.shape[0]) * 100 for dist in distances}

# Print the results
for ff, stats in ff_percentages.items():
    print(f"Freight Forwarder {ff}:")
    for dist, perc in stats.items():
        print(f"  Tours under {dist} km: {perc:.2f}%")

### Meta data 

In [ ]:
# Calculation of general meta data
meta_data = dp.calculate_meta_data(df_trips_unfiltered, df_trips, df_fleet)
# Converting meta data to DataFrame
general_df, _, _, _ = dp.meta_data_to_df(meta_data)

# Display of DataFrame
display(general_df)

# Converting meta data to DataFrame
_, temporal_df, _, _ = dp.meta_data_to_df(meta_data)

# Display of DataFrame
display(temporal_df)

# Converting meta data to DataFrame
_, _, _, ff_df = dp.meta_data_to_df(meta_data)

# Display of DataFrame
display(ff_df)

## Plots

### Data Recordings - Weekly Distance    

In [ ]:
dv.plot_weekly_distances(df_trips)  

### KDE Plots - Tracks - Distance | Duration | Max Speed | Avg Speed

In [ ]:
dv.plot_kde_plots(df_trips)

### KDE Plots - Tours - Distance | Duration | Max Speed | Avg Speed

In [ ]:
dv.plot_tour_kde_plots(df_trips)

### Weekly Distance Boxplots    

In [ ]:
dv.plot_weekly_distance_boxplot(df_trips)

dv.verify_weekday_aggregation(df_trips)

### Tour Duration and Distance  


In [ ]:
dv.plot_tour_duration_distance_histogram(df_trips, max_distance=900, max_duration=35)
dv.plot_tour_duration_distance_ecdf(df_trips, max_distance=1300, max_duration=35)

In [ ]:
dv.plot_rest_time_kde(df_stops)

## Data Quality
Gaps in recording in relation to recorded distance

In [ ]:
total_signal_loss = df_trips['n_signal_loss'].sum()
avg_signal_loss_ratio = df_trips['r_signal_loss'].mean()

print(f'Total signal loss: {total_signal_loss}')
print(f'Average signal loss ratio: {avg_signal_loss_ratio}')

## Fleet occupation 

In [ ]:
dv.plot_fleet_occupation(df_rt_joined_plot, truck_day_occ)

### Average departure and arrival times

In [ ]:
dp.avg_departure_and_arrival()